# CDX synthétique 5 ans

Le portefeuille comprend 125 noms équipondérés et le taux de recouvrement est de 60 %. Les tranches 4–9 % et 17–22 % sont valorisées hors calibration.

Pour réexécuter les cellules, garder ce notebook dans le dossier `Application-CDX-5Y`, à côté de `application.py`, `market.py` et `models.py`.


## Calibration et valorisation

**A** impose $Aq=b$. **A régularisé** pénalise cet écart avec un poids $10^5 I$. **A++** ajoute à A régularisé la contrainte calendaire dure $Dq\ge0$. **A++ ancré** ajoute une pénalité de poids $10^6 I$ sur $Cq-m$, avec $m_i=(1-R)(1-S(t_i))$.

Les quatre modèles sont calibrés avec le **prior Andersen** (corrélation LHP ajustée sur les cinq upfronts) puis avec le **prior uniforme** sur les 126 nombres de défauts. L'écart aux cotations est le maximum absolu en pb d'upfront.

La **base correlation** utilise la même courbe de survie et les mêmes cinq upfronts. Le bootstrap résout successivement une équation de repricing par détachement, de 3 % à 30 %. Pour une tranche $[a,d]$, on soustrait les jambes equity $[0,a]$ de celles de $[0,d]$, chacune à sa corrélation, puis on divise par $d-a$. Pour les bespoke, l'interpolation se fait entre strikes sur la courbe de base correlation.

Le prior Andersen utilise une seule corrélation LHP, ajustée par moindres carrés.


In [1]:
from html import escape

from IPython.display import HTML, display

def show_table(title, headers, rows, *, text_columns=1):
    """Render aligned columns with explicit headings in notebooks."""
    def tag(name, value, index):
        align = "left" if index < text_columns else "right"
        return f'<{name} style="text-align:{align};white-space:nowrap">{escape(str(value))}</{name}>'

    header = "".join(tag("th", item, i) for i, item in enumerate(headers))
    body = "".join("<tr>" + "".join(tag("td", item, i) for i, item in enumerate(row)) + "</tr>"
                   for row in rows)
    display(HTML(f"<h4>{escape(title)}</h4><div style='overflow-x:auto'>"
                 f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))

import numpy as np
from application import BENCHMARKS, TARGETS, CdxApplication
from models import EntropicCalibration, calendar_gaps

case = CdxApplication()
target_kernels = case.kernels(TARGETS)
reference = np.array([case.reference(t).upfront for t in TARGETS])


show_table("Cotations synthétiques", ("Tranche", "Coupon (pb)", "Upfront clean"),
           [(t.name, f"{t.coupon * 1e4:.0f}", f"{u:.5%}") for t, u in zip(BENCHMARKS, case.quotes)])
show_table("Prior Andersen", ("Paramètre", "Valeur"),
           [("Corrélation LHP ajustée", f"{case.lhp_fit.correlation:.5%}")])
q_ref = case.reference_marginals
assert np.min(calendar_gaps(q_ref)) >= -1e-12
np.testing.assert_allclose(q_ref[1:] @ case.portfolio.loss, case.mean_loss, atol=1e-14)
for tranche, kernel in zip((*BENCHMARKS, *TARGETS), (*case.quote_kernels, *target_kernels)):
    np.testing.assert_allclose(kernel.upfront(q_ref[1:]), case.reference(tranche).upfront,
                               atol=3e-13, rtol=0)

Tranche,Coupon (pb),Upfront clean
0-3 %,500,31.82669%
3-7 %,100,3.55443%
7-10 %,100,3.40913%
10-15 %,100,2.24797%
15-30 %,100,-4.23275%


Paramètre,Valeur
Corrélation LHP ajustée,33.02762%


In [2]:
runs = tuple(case.calibrate())
assert len(runs) == 8

base = case.base_correlation()
base_prices = np.array([base.legs(t).upfront for t in TARGETS])
base_quotes = np.array([base.legs(t).upfront for t in BENCHMARKS])
base_error = np.max(np.abs(base_quotes - case.quotes)) * 1e4
assert base_error < 1e-5

def quote_row(model, prior, prices, quote_error=None):
    errors = np.abs(prices - reference) * 1e4
    return (model, prior, *(f"{u:.5%}" for u in prices), *(f"{e:.3f}" for e in errors),
            "—" if quote_error is None else f"{quote_error:.4f}")

rows = [quote_row("Référence synthétique", "—", reference),
        quote_row("Base correlation", "—", base_prices, base_error)]
anchor_rows = []
for run in runs:
    q = run.marginals
    prices = np.array([k.upfront(q) for k in target_kernels])
    error = np.max(np.abs(np.array([k.upfront(q) for k in case.quote_kernels]) - case.quotes)) * 1e4
    rows.append(quote_row(run.name, run.prior, prices, error))
    assert np.all(q >= -1e-14) and np.allclose(q.sum(axis=1), 1)
    if run.name == "A":
        assert error < 1e-5
    if run.name.startswith("A++"):
        assert np.min(calendar_gaps(q)) >= -1e-12
        assert abs(run.fit.primal_dual_gap) < 3e-6
    if run.name == "A++ ancré":
        gap = np.max(np.abs(q @ case.portfolio.loss - case.mean_loss)) * 1e4
        anchor_rows.append((run.prior, f"{gap:.3f}"))

show_table("Upfronts hors calibration et écarts à la référence",
           ("Modèle", "Prior", "Upfront 4–9 %", "Upfront 17–22 %",
            "|Erreur| 4–9 (pb)", "|Erreur| 17–22 (pb)", "Écart max. cotations (pb)"),
           rows, text_columns=2)
show_table("Base correlation par détachement", ("Détachement", "Corrélation"),
           [(f"{k:.0%}", f"{rho:.4%}") for k, rho in zip(base.detachments, base.correlations)])
show_table("Ancrage sur la perte moyenne", ("Prior", "Écart max. (pb d’indice)"), anchor_rows)

Modèle,Prior,Upfront 4–9 %,Upfront 17–22 %,|Erreur| 4–9 (pb),|Erreur| 17–22 (pb),Écart max. cotations (pb)
Référence synthétique,—,3.41516%,-4.42115%,0.000,0.000,—
Base correlation,—,2.51285%,-4.17956%,90.231,24.159,0.0000
A,Andersen,3.44623%,-4.34310%,3.108,7.805,0.0000
A régularisé,Andersen,3.44249%,-4.34380%,2.733,7.735,1.7365
A++,Andersen,3.44425%,-4.30803%,2.909,11.312,1.6221
A++ ancré,Andersen,3.43867%,-4.23617%,2.352,18.498,1.6968
A,Uniforme,3.42116%,-4.33922%,0.601,8.192,0.0000
A régularisé,Uniforme,3.40942%,-4.28458%,0.573,13.657,3.8841
A++,Uniforme,3.42480%,-4.20116%,0.965,21.999,2.1394
A++ ancré,Uniforme,3.43715%,-4.26947%,2.199,15.168,2.0235


Détachement,Corrélation
3%,31.9834%
7%,48.6064%
10%,49.1794%
15%,35.8040%
30%,57.3448%


Prior,Écart max. (pb d’indice)
Andersen,0.131
Uniforme,0.380


## Diagnostic des violations calendaires de A et A++

Pour chaque prior, on teste les marginales utilisées pour le repricing. Une violation est comptée lorsque la probabilité de dépasser un seuil de perte diminue entre deux dates consécutives de calibration, au-delà d'une tolérance de $10^{-12}$. La baisse maximale est exprimée en points de base de probabilité.


In [3]:
calendar_tolerance = 1e-12
rows = []
for run in runs:
    if run.name not in ("A", "A++"):
        continue
    gaps = calendar_gaps(run.marginals)
    assert np.isfinite(gaps).all()
    violations = gaps < -calendar_tolerance
    if violations.any():
        i, j = np.unravel_index(np.argmin(gaps), gaps.shape)
        times = case.grid.times[1:]
        worst_interval = f"{times[i]:.3f} -> {times[i + 1]:.3f}"
        threshold = f"{case.portfolio.loss[j]:.3%}"
        drop = f"{-gaps[i, j] * 1e4:.3f}"
    else:
        worst_interval = threshold = "—"
        drop = "0.000"
    rows.append((run.name, run.prior, f"{violations.sum()} / {gaps.size}",
                 f"{violations.any(axis=1).sum()} / {gaps.shape[0]}",
                 drop, worst_interval, threshold))

show_table("Violations calendaires après repricing",
           ("Modèle", "Prior", "Contraintes violées", "Transitions touchées",
            "Baisse max. (pb de prob.)", "Pire intervalle (années)", "Seuil de perte"),
           rows, text_columns=2)


Modèle,Prior,Contraintes violées,Transitions touchées,Baisse max. (pb de prob.),Pire intervalle (années),Seuil de perte
A,Andersen,75 / 2500,1 / 20,376.418,4.855 -> 5.104,2.880%
A++,Andersen,0 / 2500,0 / 20,0.000,—,—
A,Uniforme,750 / 2500,6 / 20,4578.486,4.855 -> 5.104,1.920%
A++,Uniforme,0 / 2500,0 / 20,0.000,—,—


## Sensibilité des upfronts à un choc de +1 pb

On augmente une seule cotation de 1 pb et on recalibre le modèle. La dernière colonne indique la variation de l’upfront de la tranche en pb. La colonne analytique donne la dérivée locale en pb d’upfront par pb de cotation. Pour la base correlation, elle vient de la différentiation implicite du bootstrap, suivie de celle de l’interpolation linéaire. Pour A++ et A++ ancré, le tiret indique que les conditions numériques de la formule analytique ne sont pas réunies.


In [4]:
from dataclasses import replace
from statistics import NormalDist

from models import BaseCorrelation


def equity_upfront_rho(strike, rho, coupon):
    """Derivative of the equity contribution per unit of portfolio notional."""
    if strike == 0:
        return 0.0
    lhp = case.lhp
    recovery = case.portfolio.recovery

    def cap_rho(amount, cap):
        if not 0 < cap < amount:
            return np.zeros_like(lhp.pd)
        quantile = NormalDist().inv_cdf(cap / amount)
        split = (lhp.thresholds - np.sqrt(1 - rho) * quantile) / np.sqrt(rho)
        # Plackett's identity for the bivariate normal CDF.
        density = np.exp(-0.5 * (split**2 + quantile**2)) / (2 * np.pi)
        return np.r_[0.0, -amount * density / (2 * np.sqrt(rho * (1 - rho)))]

    loss_rho = cap_rho(1 - recovery, strike)
    reduction_rho = loss_rho - cap_rho(recovery, 1 - strike)
    grid = case.grid
    return (grid.protection_weights @ loss_rho
            + coupon * (grid.premium_weights @ reduction_rho)) / grid.settlement_discount


def base_quote_sensitivity(tranche, quote_direction):
    """Differentiate the sequential bootstrap, then interpolate its derivative."""
    correlation_slope = []
    previous_rho, previous_slope = 0.0, 0.0
    for quoted, rho, quote_slope in zip(BENCHMARKS, base.correlations, quote_direction):
        width = quoted.detachment - quoted.attachment
        lower = equity_upfront_rho(quoted.attachment, previous_rho, quoted.coupon)
        upper = equity_upfront_rho(quoted.detachment, rho, quoted.coupon)
        if abs(upper) <= 1e-12:
            raise ValueError(f"Bootstrap derivative is singular for {quoted.name}.")
        previous_slope = (width * quote_slope + lower * previous_slope) / upper
        correlation_slope.append(previous_slope)
        previous_rho = rho

    def equity_slope(strike):
        rho_slope = np.interp(strike, base.detachments, correlation_slope)
        return equity_upfront_rho(strike, base.rho(strike), tranche.coupon) * rho_slope

    return (equity_slope(tranche.detachment) - equity_slope(tranche.attachment)) / (
        tranche.detachment - tranche.attachment
    )


def analytical_quote_slope(run, tranche, quote_direction):
    multipliers = run.fit.calendar_multiplier
    if multipliers.size and not np.any(multipliers > 1e-7):
        gaps = calendar_gaps(run.fit.q)
        status = "Ensemble actif ambigu" if np.any(gaps <= 3e-7) else "Aucune contrainte active"
        return None, status
    try:
        return case.sensitivity(run, (tranche,), upfront=quote_direction)[0], ""
    except ValueError as exc:
        if str(exc).startswith("Weak calendar constraints"):
            return None, "Ensemble actif ambigu"
        if str(exc).startswith("Active calendar residual"):
            return None, "Résidu calendaire trop élevé"
        raise
    except RuntimeError as exc:
        if str(exc).startswith("Active-set sensitivity solve did not converge"):
            return None, "Système linéarisé non résolu"
        raise


bump_step = 1e-4
quote_directions = np.eye(len(BENCHMARKS))
for tranche in TARGETS:
    kernel = case.kernels((tranche,))[0]
    for index in case.overlapping_quote_indices(tranche):
        direction = quote_directions[index]
        bumped_base = BaseCorrelation(case.lhp, BENCHMARKS, case.quotes + bump_step * direction)
        base_change = (bumped_base.legs(tranche).upfront - base.legs(tranche).upfront) * 1e4
        rows = [("Base correlation", "—", f"{base_quote_sensitivity(tranche, direction):+.5f}",
                 f"{base_change:+.5f}")]

        for run in runs:
            analytical, status = analytical_quote_slope(run, tranche, direction)
            rhs_change = case._rhs_direction(run, direction, None)
            bumped_problem = run.problem.refit(rhs=run.problem.b + bump_step * rhs_change)
            bumped = replace(run, problem=bumped_problem, fit=bumped_problem.solve())
            change = (kernel.upfront(bumped.marginals) - kernel.upfront(run.marginals)) * 1e4
            assert np.isfinite(change)
            analytical_cell = f"{analytical:+.5f}" if analytical is not None else f"— ({status.lower()})"
            rows.append((run.name, run.prior, analytical_cell, f"{change:+.5f}"))

        assert len(rows) == 9
        show_table(f"{tranche.name} · +1 pb sur {BENCHMARKS[index].name}",
                   ("Modèle", "Prior", "Analytique (pb/pb)", "Variation recalibrée (pb)"),
                   rows, text_columns=2)


Modèle,Prior,Analytique (pb/pb),Variation recalibrée (pb)
Base correlation,—,+0.59824,+0.59824
A,Andersen,+0.37419,+0.37843
A régularisé,Andersen,+0.41204,+0.41414
A++,Andersen,— (ensemble actif ambigu),+0.43407
A++ ancré,Andersen,— (ensemble actif ambigu),+0.31464
A,Uniforme,+0.15993,+0.16491
A régularisé,Uniforme,+0.26135,+0.26303
A++,Uniforme,— (ensemble actif ambigu),+0.11728
A++ ancré,Uniforme,— (ensemble actif ambigu),+0.28520


Modèle,Prior,Analytique (pb/pb),Variation recalibrée (pb)
Base correlation,—,+0.42455,+0.42458
A,Andersen,+0.62616,+0.63066
A régularisé,Andersen,+0.58826,+0.59040
A++,Andersen,— (ensemble actif ambigu),+0.57206
A++ ancré,Andersen,— (ensemble actif ambigu),+0.68428
A,Uniforme,+0.84836,+0.85289
A régularisé,Uniforme,+0.74432,+0.74551
A++,Uniforme,— (ensemble actif ambigu),+0.87498
A++ ancré,Uniforme,— (ensemble actif ambigu),+0.71377


Modèle,Prior,Analytique (pb/pb),Variation recalibrée (pb)
Base correlation,—,+1.64581,+1.67051
A,Andersen,+1.31259,+1.32644
A régularisé,Andersen,+1.30831,+1.32220
A++,Andersen,— (ensemble actif ambigu),+1.01926
A++ ancré,Andersen,— (ensemble actif ambigu),+1.78498
A,Uniforme,+1.38114,+1.39148
A régularisé,Uniforme,+1.35069,+1.35930
A++,Uniforme,— (ensemble actif ambigu),+1.45284
A++ ancré,Uniforme,— (ensemble actif ambigu),+0.27498
